# Advanced extension: regularised depletion Poisson vs equilibrium Poisson-Boltzmann

The first model smooths a prescribed depletion charge for numerical purposes.
The second computes mobile carriers self-consistently at thermal equilibrium.
It uses Boltzmann statistics, complete ionisation, an abrupt doping step, and
the Einstein relation. The extension is not a drift-diffusion device simulator.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_bvp
from scipy.interpolate import interp1d
from scipy.optimize import least_squares
from scipy.sparse import diags

q = 1.602176634e-19
eps0 = 8.8541878128e-12
eps_si = 11.7*eps0
kB = 1.380649e-23

def depletion_edges(Na_cm3, Nd_cm3, Vbi):
    Na, Nd = Na_cm3*1e6, Nd_cm3*1e6
    d = np.sqrt(2*eps_si*Vbi/q*(Na+Nd)/(Na*Nd))
    return d, Nd*d/(Na+Nd), Na*d/(Na+Nd)

def debye_lengths(Na_cm3, Nd_cm3, T):
    Na, Nd = Na_cm3*1e6, Nd_cm3*1e6
    return (np.sqrt(eps_si*kB*T/(q*q*Na)),
            np.sqrt(eps_si*kB*T/(q*q*Nd)))

def smooth_window(x, left, right, delta):
    return 0.5*(np.tanh((x-left)/delta)-np.tanh((x-right)/delta))

def regularised_depletion(Na_cm3, Nd_cm3, T=300., ni_cm3=1e10,
                          buffer_factor=5., points=1201,
                          delta_fraction=0.01):
    Na, Nd = Na_cm3*1e6, Nd_cm3*1e6
    VT = kB*T/q
    Vbi = VT*np.log(Na_cm3*Nd_cm3/ni_cm3**2)
    d, xp, xn = depletion_edges(Na_cm3, Nd_cm3, Vbi)
    LDp, LDn = debye_lengths(Na_cm3, Nd_cm3, T)
    x = np.linspace(-xp-buffer_factor*LDp, xn+buffer_factor*LDn, points)
    delta = delta_fraction*d
    rho = q*(Nd*smooth_window(x, 0., xn, delta)
             - Na*smooth_window(x, -xp, 0., delta))
    u = x/d
    def fun(us, y):
        xs = us*d
        rs = q*(Nd*smooth_window(xs, 0., xn, delta)
                - Na*smooth_window(xs, -xp, 0., delta))
        return np.vstack((y[1], -d*d*rs/eps_si))
    def bc(ya, yb):
        return np.array([ya[0], yb[0]-Vbi])
    guess = np.vstack((Vbi*(u-u[0])/(u[-1]-u[0]),
                       np.full_like(u, Vbi/(u[-1]-u[0]))))
    sol = solve_bvp(fun, bc, u, guess, tol=2e-6, max_nodes=30000)
    if not sol.success:
        raise RuntimeError(sol.message)
    phi = sol.sol(u)[0]
    E = -sol.sol(u)[1]/d
    return dict(x=x, phi=phi, E=E, rho=rho, Vbi=Vbi, d=d, xp=xp, xn=xn,
                delta_num=delta, success=sol.success)


def poisson_boltzmann(Na_cm3, Nd_cm3, T=300., ni_cm3=1e10,
                      buffer_factor=12., points=1601):
    """Equilibrium, non-degenerate, complete-ionisation p-n junction."""
    Na, Nd, ni = Na_cm3*1e6, Nd_cm3*1e6, ni_cm3*1e6
    VT = kB*T/q
    psi_p = -np.arcsinh(Na/(2*ni))
    psi_n =  np.arcsinh(Nd/(2*ni))
    Vbi = VT*(psi_n-psi_p)
    d, xp, xn = depletion_edges(Na_cm3, Nd_cm3, Vbi)
    LDp, LDn = debye_lengths(Na_cm3, Nd_cm3, T)
    x = np.linspace(-xp-buffer_factor*LDp, xn+buffer_factor*LDn, points)
    L0 = max(d, LDp, LDn)
    u = x/L0
    def dopants(xs):
        return np.where(xs < 0., -Na, Nd)
    # The depletion solution is a physically informed initial guess and makes
    # the stiff exponential boundary-value problem robust for asymmetric cases.
    dep0 = regularised_depletion(Na_cm3, Nd_cm3, T, ni_cm3,
                                  buffer_factor=buffer_factor, points=points)
    phi0 = interp1d(dep0["x"], dep0["phi"], kind="cubic",
                    fill_value="extrapolate")(x)
    psi_guess = psi_p + phi0/VT
    h = u[1]-u[0]
    C = L0*L0*q/(VT*eps_si)
    dop_i = dopants(x[1:-1])
    def residual(z):
        psi = np.r_[psi_p, z, psi_n]
        pc = ni*np.exp(-np.clip(psi[1:-1], -80., 80.))
        nc = ni*np.exp( np.clip(psi[1:-1], -80., 80.))
        return (psi[:-2]-2*psi[1:-1]+psi[2:]
                + h*h*C*(pc-nc+dop_i))
    m = points-2
    sparsity = diags([np.ones(m-1),np.ones(m),np.ones(m-1)],[-1,0,1],
                     shape=(m,m),format="csr")
    fit = least_squares(residual, psi_guess[1:-1], jac_sparsity=sparsity,
                        xtol=1e-10, ftol=1e-10, gtol=1e-10,
                        max_nfev=5000)
    if not fit.success or np.max(np.abs(residual(fit.x))) > 1e-5:
        raise RuntimeError("Poisson-Boltzmann nonlinear solve did not converge")
    psi = np.r_[psi_p, fit.x, psi_n]
    dpsi_du = np.gradient(psi, u, edge_order=2)
    phi = VT*(psi-psi[0])
    E = -VT*dpsi_du/L0
    p, n = ni*np.exp(-psi), ni*np.exp(psi)
    rho = q*(p-n+dopants(x))
    # Electrochemical equilibrium: d ln(n)/dx=qE/(kT), d ln(p)/dx=-qE/(kT).
    dn_dx = np.gradient(n, x, edge_order=2)
    dp_dx = np.gradient(p, x, edge_order=2)
    mu_n, mu_p = 0.135, 0.048
    Dn, Dp = mu_n*VT, mu_p*VT
    Jn = q*mu_n*n*E + q*Dn*dn_dx
    Jp = q*mu_p*p*E - q*Dp*dp_dx
    current_scale = q*max(mu_n, mu_p)*max(np.max(n), np.max(p))*max(np.max(np.abs(E)), 1.)
    return dict(x=x, phi=phi, E=E, rho=rho, n=n, p=p, Jn=Jn, Jp=Jp,
                Vbi=Vbi, d=d, xp=xp, xn=xn, current_residual=
                max(np.max(np.abs(Jn)), np.max(np.abs(Jp)))/current_scale,
                nonlinear_residual=np.max(np.abs(residual(fit.x))),
                success=fit.success)


In [ ]:
Na, Nd, T, ni = 1e17, 1e16, 300., 1e10
dep = regularised_depletion(Na, Nd, T, ni, buffer_factor=12., points=1601)
pb = poisson_boltzmann(Na, Nd, T, ni)
print(f"depletion drop = {dep['phi'][-1]-dep['phi'][0]:.6f} V")
print(f"PB drop        = {pb['phi'][-1]-pb['phi'][0]:.6f} V")
print(f"dimensionless zero-current residual = {pb['current_residual']:.3e}")

fig, ax = plt.subplots(1,3,figsize=(15,4))
ax[0].plot(dep['x']*1e6, dep['phi'], label='regularised depletion')
ax[0].plot(pb['x']*1e6, pb['phi'], '--', label='Poisson-Boltzmann')
ax[0].set(xlabel='x (um)', ylabel='phi (V)')
ax[1].plot(dep['x']*1e6, dep['E']/1e5)
ax[1].plot(pb['x']*1e6, pb['E']/1e5, '--')
ax[1].set(xlabel='x (um)', ylabel='E (kV/cm)')
ax[2].plot(dep['x']*1e6, dep['rho']/q/1e6)
ax[2].plot(pb['x']*1e6, pb['rho']/q/1e6, '--')
ax[2].set(xlabel='x (um)', ylabel='net charge / q (cm^-3)')
for a in ax: a.grid(); a.legend()
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(7,4))
ax.semilogy(pb['x']*1e6, pb['n']/1e6, label='n')
ax.semilogy(pb['x']*1e6, pb['p']/1e6, label='p')
ax.set(xlabel='x (um)', ylabel='carrier density (cm^-3)')
ax.grid(); ax.legend(); plt.tight_layout(); plt.show()

## Questions

1. Identify the regions where mobile carriers screen the ionised dopants.
2. Explain why changing `delta_fraction` affects only the regularised model.
3. Use the printed current residual to discuss zero-net-current equilibrium.
4. State the assumptions that would fail under applied bias.